<a href="https://colab.research.google.com/github/rfcastrovera/BIGDATA/blob/main/Lab6_NYC_Taxi_Streaming_Kafka.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 6: Primer pipeline de streaming — Kafka, Structured Streaming y ventanas sobre NYC Taxi

**Análisis de Big Data · Magíster en Data Science UDD · Sesión 6 (lunes 14 de septiembre)**

Todo lo hecho hasta el Lab 5 procesa datos que ya llegaron. Desde hoy, datos que están llegando. El
laboratorio vuelve a NYC Taxi —el mismo Parquet de enero de 2023 de los Labs 1 a 4— pero lo trata como
lo que fue en su momento: **un flujo de eventos** (*stream*), un viaje a la vez. Un *productor* relee un
día de viajes y los publica como eventos JSON en un tópico de Kafka con el tiempo acelerado (una hora de
viajes cada cinco segundos); un *consumidor* en Structured Streaming los agrega por **ventanas de tiempo
de evento** y muestra una tabla que crece mientras se mira.

Kafka corre **dentro de Colab**, en un solo nodo, levantado por la celda 0. Si el *broker* no responde en
90 segundos, la misma celda cambia a una **fuente de archivos** y todo lo demás sigue igual: desde la
sección 2 en adelante el notebook es idéntico en las dos rutas. Esa es la primera lección del lab: la
fuente es un detalle de conexión; la semántica —ventanas, tiempo de evento, modos de salida— es la misma.

---

## Cómo se trabaja hoy

- **Celda 0.** Descarga Kafka, baja el conector de Spark y levanta el *broker*: de
  tres a cinco minutos. A las 20:55 el entorno está listo sin que nadie haya esperado por él.
- **En sala (20:55–21:25):** secciones 1 a 4 — el productor en marcha y la **consulta A** con la tabla
  poblada. Esa es la meta.
- **En casa:** secciones 5 a 8 — consultas B y C, detener y mirar el *checkpoint*, y el entregable.
- **Extensión:** sección 9, propuesta y no evaluada.

---

## Qué se entrega — miércoles 23 de septiembre, 23:59

**No se entrega el notebook completo.** Se entrega:

1. La **tabla de una agregación por ventana** (consulta A o B, sección 8): al menos diez filas, con la
   ventana y la zona visibles.
2. **Un párrafo** (sección 8b) que diga **qué reloj** usa esa ventana, **qué pasó con los eventos que
   llegaron tarde** y **qué cambia** entre la ventana de salto fijo y la deslizante.

Menos de dos horas de trabajo para quien estuvo en clase. La Fase 2 vence el lunes 21; el Lab 6 vence
dos días después, a propósito. La versión resuelta se publica el jueves 24.

---

## Objetivos

- Distinguir **tiempo de evento** (*event time*) de **tiempo de procesamiento** (*processing time*) sobre
  los mismos viajes, y comprobar que dan tablas distintas.
- Publicar y consumir eventos en un tópico de Kafka: tópico, partición, llave, *offset*, y por qué un
  consumidor que arranca tarde puede leer desde el principio (*replay*).
- Escribir una agregación por **ventana de salto fijo** (*tumbling*) y por **ventana deslizante**
  (*sliding*) con `readStream`, `window()` y `writeStream`, y elegir el **modo de salida**.
- Reconocer, con los propios datos, el problema que la sesión 7 resuelve: sin *watermark*, una ventana
  nunca se cierra y el estado crece sin límite.

## 0. Celda 0

Una sola celda hace cuatro cosas: **(a)** instala PySpark 3.5.1 y el cliente Python de Kafka; **(b)**
descarga Kafka 3.7 (unos 120 MB) y lo levanta en modo **KRaft** de un solo nodo —sin ZooKeeper, con
512 MB de *heap* para convivir con Spark en los 12 GB de Colab—; **(c)** crea el tópico `viajes` con
tres particiones; **(d)** crea la `SparkSession` con el conector `spark-sql-kafka-0-10`, que baja de
Maven.

Al terminar imprime una de dos líneas, y **las dos son válidas para el
laboratorio**:

- `KAFKA: OK` — el *broker* responde en `localhost:9092`.
- `KAFKA: no disponible → fuente de archivos` — el productor escribirá los mismos eventos como archivos
  JSON en `/content/stream_in/` y el consumidor los leerá con `readStream.format("json")`.

Dos ajustes que hacen la diferencia entre una demo que funciona y una que no:
`spark.sql.shuffle.partitions=4` (con las 200 por defecto cada micro-lote tarda segundos que se sienten)
y el *heap* de Kafka acotado. Si en sala algo del entorno falla, no se depura más de dos minutos:
`USAR_KAFKA = False` y se sigue.

In [1]:
# ===== CELDA 0 =====
import os, sys, time, socket, subprocess, shutil, ssl, tarfile, urllib.request
print("Python", ".".join(map(str, sys.version_info[:3])))
t0 = time.time()

# (a) PySpark fijado en 3.5.1 (la misma combinación de los Labs 4 y 5) y el cliente Python de Kafka.
!pip -q install --only-binary=:all: pyarrow
!pip -q install --prefer-binary pyspark==3.5.1 "kafka-python>=2.1"

# --- Descargas que no fallan en silencio (lección del Lab 5: wget -q dejaba un archivo de error con nombre de zip) ---
def descargar(url, destino, verificar_ssl=True, timeout=120):
    contexto = None if verificar_ssl else ssl._create_unverified_context()
    pedido = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(pedido, timeout=timeout, context=contexto) as r, open(destino, "wb") as f:
        shutil.copyfileobj(r, f)

def obtener(destino, urls, valido, enlace_drive=""):
    """Prueba cada URL con y sin verificación de certificado, después el respaldo de Drive. Borra lo que no valide."""
    if valido(destino):
        return True
    for url in urls:
        for verificar in (True, False):
            if os.path.exists(destino):
                os.remove(destino)
            try:
                descargar(url, destino, verificar)
                if valido(destino):
                    print(f"descargado ({os.path.getsize(destino)/1e6:.0f} MB): {url}")
                    return True
            except Exception as e:
                print(f"falló {url} ({'con' if verificar else 'sin'} verificación): {type(e).__name__}")
    if enlace_drive:
        if os.path.exists(destino):
            os.remove(destino)
        import gdown
        gdown.download(url=enlace_drive, output=destino, quiet=True, fuzzy=True)
        if valido(destino):
            print("descargado desde el respaldo de Drive")
            return True
    if os.path.exists(destino):
        os.remove(destino)
    return False

# (b) Kafka 3.7.1 en modo KRaft, un solo nodo. Archivo de Apache, espejo y respaldo en Drive.
KAFKA_VERSION = "3.7.1"
KAFKA_DIR     = f"/content/kafka_2.13-{KAFKA_VERSION}"
KAFKA_TGZ     = f"/content/kafka_2.13-{KAFKA_VERSION}.tgz"
URLS_KAFKA    = [f"https://archive.apache.org/dist/kafka/{KAFKA_VERSION}/kafka_2.13-{KAFKA_VERSION}.tgz",
                 f"https://dlcdn.apache.org/kafka/{KAFKA_VERSION}/kafka_2.13-{KAFKA_VERSION}.tgz"]
ENLACE_RESPALDO_KAFKA = ""   # enlace de Drive al mismo .tgz (receta del Lab 4), por si Apache no responde
BOOTSTRAP  = "localhost:9092"
TOPICO     = "viajes"
USAR_KAFKA = True            # forzar False para probar la ruta de archivos

tgz_valido = lambda ruta: os.path.exists(ruta) and os.path.getsize(ruta) > 50_000_000 and tarfile.is_tarfile(ruta)

def puerto_abierto(host, puerto, timeout=1.0):
    try:
        with socket.create_connection((host, puerto), timeout=timeout):
            return True
    except OSError:
        return False

try:
    if shutil.which("java") is None:
        raise RuntimeError("no hay Java en el entorno")
    if not os.path.isdir(KAFKA_DIR):
        if not obtener(KAFKA_TGZ, URLS_KAFKA, tgz_valido, ENLACE_RESPALDO_KAFKA):
            raise RuntimeError("no se pudo descargar Kafka")
        with tarfile.open(KAFKA_TGZ) as tgz:
            tgz.extractall("/content")
    if not os.path.isdir(KAFKA_DIR):
        raise RuntimeError("la descompresión de Kafka no se completó")
    print(f"Kafka descargado y descomprimido ({time.time() - t0:.0f} s)")

    if not puerto_abierto("localhost", 9092):
        # Formatear el almacenamiento KRaft (una sola vez) y arrancar el broker como demonio, con heap acotado.
        cfg = f"{KAFKA_DIR}/config/kraft/server.properties"
        if not os.path.exists("/tmp/kraft-combined-logs/meta.properties"):
            uuid = subprocess.check_output([f"{KAFKA_DIR}/bin/kafka-storage.sh", "random-uuid"], text=True).strip()
            subprocess.run([f"{KAFKA_DIR}/bin/kafka-storage.sh", "format", "-t", uuid, "-c", cfg],
                           check=True, capture_output=True)
        env = dict(os.environ, KAFKA_HEAP_OPTS="-Xmx512M -Xms256M")
        subprocess.run([f"{KAFKA_DIR}/bin/kafka-server-start.sh", "-daemon", cfg], env=env, check=True)

    # Red de seguridad: hasta 90 segundos esperando el puerto 9092.
    t1 = time.time()
    while not puerto_abierto("localhost", 9092) and time.time() - t1 < 90:
        time.sleep(3)
    if not puerto_abierto("localhost", 9092):
        raise RuntimeError("el broker no respondió en 90 s")

    # (c) El tópico: 3 particiones, 1 réplica (un solo nodo). La llave de cada mensaje decidirá la partición.
    subprocess.run([f"{KAFKA_DIR}/bin/kafka-topics.sh", "--create", "--if-not-exists", "--topic", TOPICO,
                    "--partitions", "3", "--replication-factor", "1", "--bootstrap-server", BOOTSTRAP],
                   check=True, capture_output=True, timeout=90)
    USAR_KAFKA = True
except Exception as e:
    USAR_KAFKA = False
    print("Kafka no disponible. Motivo:", repr(e)[:200])

# (d) SparkSession con el conector de Kafka (baja de Maven, alrededor de un minuto la primera vez).
from pyspark.sql import SparkSession, functions as F, types as T
spark = (SparkSession.builder
         .appName("lab6-streaming")
         .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.1")
         .config("spark.sql.shuffle.partitions", "4")     # con las 200 por defecto cada micro-lote tardaría segundos de más
         .config("spark.driver.memory", "4g")              # deja espacio al broker de Kafka en los 12 GB de Colab
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

# Comprobación del conector: una lectura por lotes del tópico (vacío todavía). Si el jar no bajó de Maven, se sabe aquí y no en la consulta A.
if USAR_KAFKA:
    try:
        n_msgs = (spark.read.format("kafka").option("kafka.bootstrap.servers", BOOTSTRAP)
                  .option("subscribe", TOPICO).load().count())
        print(f"conector de Kafka OK · mensajes ya en el tópico: {n_msgs}")
    except Exception as e:
        USAR_KAFKA = False
        print("Conector de Kafka no disponible. Motivo:", repr(e)[:200])

RUTA_ARCHIVOS = "/content/stream_in"     # ruta de respaldo: el productor escribe aquí si no hay Kafka
RUTA_CHK      = "/content/chk"           # directorios de checkpoint, uno por consulta
os.makedirs(RUTA_ARCHIVOS, exist_ok=True); os.makedirs(RUTA_CHK, exist_ok=True)

print("Spark", spark.version, f"| celda 0 completa en {time.time() - t0:.0f} s")
print("KAFKA: OK" if USAR_KAFKA else "KAFKA: no disponible → fuente de archivos")

Python 3.13.15
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.0/317.0 MB 4.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.5/200.5 kB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 614.2/614.2 kB 28.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dataproc-spark-connect 1.1.0 requires pyspark[connect]~=4.0.0, but you have pyspark 3.5.1 which is incompatible.
descargado (120 MB): https://archive.apache.org/dist/kafka/3.7.1/kafka_2.13-3.7.1.tgz


/tmp/ipykernel_4260/2538237517.py:71: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tgz.extractall("/content")


Kafka descargado y descomprimido (80 s)
conector de Kafka OK · mensajes ya en el tópico: 0
Spark 3.5.1 | celda 0 completa en 124 s
KAFKA: OK


## 1. Los datos, esta vez como eventos

El mismo Parquet de enero de 2023 de los Labs 1 a 4 (48 MB), leído con pyarrow en el driver —es un
archivo, no un cluster—. Se toma **un día**: el martes 17 de enero, unos 100 mil viajes. Cada fila se
convierte en un evento JSON con cinco campos: `pickup_ts` (el **tiempo de evento**: cuándo empezó el
viaje), `zona` (zona de subida, `PULocationID`), `tarifa`, `distancia` y `pasajeros`, más una
marca `tardio` que el productor usa para el 5% que retrasa a propósito.

El tiempo de evento es `tpep_pickup_datetime`. El tiempo de procesamiento será el reloj de Colab en el
momento en que cada evento llegue a Spark. No coinciden, y esa diferencia es el tema de la sección 6.

In [2]:
import pandas as pd, numpy as np, json, threading, random
import pyarrow.parquet as pq

URLS_PARQUET = ["https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-01.parquet"]
RUTA_PARQUET = "/content/yellow_tripdata_2023-01.parquet"
ENLACE_RESPALDO_PARQUET = ""   # enlace de Drive al mismo Parquet (receta gdown), por si el CDN de TLC no responde
DIA = "2023-01-17"             # un martes normal de enero

def parquet_valido(ruta):
    """Un archivo a medias o una página de error no abre como Parquet."""
    if not (os.path.exists(ruta) and os.path.getsize(ruta) > 10_000_000):
        return False
    try:
        pq.ParquetFile(ruta); return True
    except Exception:
        return False

if not obtener(RUTA_PARQUET, URLS_PARQUET, parquet_valido, ENLACE_RESPALDO_PARQUET):
    raise RuntimeError("El Parquet no llegó completo. Pegar en ENLACE_RESPALDO_PARQUET el enlace de Drive y volver a ejecutar.")

cols = ["tpep_pickup_datetime", "PULocationID", "fare_amount", "trip_distance", "passenger_count"]
mes  = pq.read_table(RUTA_PARQUET, columns=cols).to_pandas()
dia  = (mes[(mes["tpep_pickup_datetime"] >= DIA) &
            (mes["tpep_pickup_datetime"] < pd.Timestamp(DIA) + pd.Timedelta(days=1))]
          .dropna()
          .query("fare_amount > 0 and trip_distance > 0")
          .rename(columns={"tpep_pickup_datetime": "pickup_ts", "PULocationID": "zona", "fare_amount": "tarifa",
                           "trip_distance": "distancia", "passenger_count": "pasajeros"})
          .astype({"zona": "int32", "pasajeros": "int32"})
          .sort_values("pickup_ts")
          .reset_index(drop=True))
del mes

print(f"viajes del {DIA}: {len(dia):,}  |  zonas distintas: {dia['zona'].nunique()}")
print(f"primer viaje {dia['pickup_ts'].min()}  ·  último {dia['pickup_ts'].max()}")
dia.head(3)

descargado (48 MB): https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-01.parquet
viajes del 2023-01-17: 96,657  |  zonas distintas: 207
primer viaje 2023-01-17 00:00:02  ·  último 2023-01-17 23:59:58


,pickup_ts,zona,tarifa,distancia,pasajeros
0,2023-01-17 00:00:02,132,33.8,8.4,2
1,2023-01-17 00:00:02,132,24.7,6.0,2
2,2023-01-17 00:00:05,70,14.2,3.2,1


## 2. El productor: un día de viajes en dos minutos

El productor relee el día en **orden de tiempo de evento** y publica cada viaje como un mensaje JSON en
el tópico `viajes`, con **la zona de subida como llave**: Kafka asigna la partición por la llave, así
que todos los viajes de una misma zona caen en la misma partición y quedan ordenados entre sí. Entre
zonas distintas el orden no se garantiza —y no hace falta—.

El tiempo va acelerado: **una hora de viajes cada cinco segundos**, en bloques de 15 minutos, así que el
día completo se publica en poco más de dos minutos y la primera ventana aparece en la tabla antes de 30
segundos. Corre en un hilo de fondo; `DETENER.set()` lo para.

**Un 5% de los viajes sale con retraso, a propósito.** Se eligen al azar y se publican dos horas de tiempo
de evento después de su turno —diez segundos de reloj—. Son los eventos tardíos (*late events*) de los
que habló la cápsula: un taxímetro sin señal que reporta el viaje más tarde. Hoy se ven; el viernes 25 se
decide qué hacer con ellos.

Si `USAR_KAFKA` es `False`, el mismo hilo escribe cada bloque como un archivo JSON-lines en
`/content/stream_in/` (escrito completo y luego renombrado, para que Spark nunca lea un archivo a
medias). Misma semántica, distinta conexión.

In [3]:
SEG_POR_HORA    = 5        # una hora de viajes cada 5 segundos → el día completo en ~2 minutos
BLOQUES_TARDIO  = 8        # el 5% tardío se publica 8 bloques de 15 min después: dos horas de evento, diez segundos de reloj
FRACCION_TARDIA = 0.05

# Cada viaje pertenece a un bloque de 15 minutos; el tardío cambia de bloque de emisión sin cambiar su pickup_ts.
dia["bloque"] = (dia["pickup_ts"].dt.hour * 4 + dia["pickup_ts"].dt.minute // 15).astype(int)
dia["tardio"] = np.random.default_rng(42).random(len(dia)) < FRACCION_TARDIA
dia["bloque_emision"] = dia["bloque"] + dia["tardio"].astype(int) * BLOQUES_TARDIO
N_BLOQUES = int(dia["bloque_emision"].max()) + 1
print(f"{dia['tardio'].sum():,} viajes tardíos de {len(dia):,} · {N_BLOQUES} bloques · duración ≈ {N_BLOQUES * SEG_POR_HORA / 4:.0f} s")

def a_evento(fila):
    return {"pickup_ts": fila.pickup_ts.strftime("%Y-%m-%d %H:%M:%S"), "zona": int(fila.zona),
            "tarifa": round(float(fila.tarifa), 2), "distancia": round(float(fila.distancia), 2),
            "pasajeros": int(fila.pasajeros), "tardio": bool(fila.tardio)}

DETENER = threading.Event()
ESTADO  = {"bloque": -1, "enviados": 0, "fin": False}

if USAR_KAFKA:
    from kafka import KafkaProducer
    productor = KafkaProducer(bootstrap_servers=BOOTSTRAP, linger_ms=20, batch_size=64 * 1024)

def _publicar_kafka(eventos):
    for ev in eventos:
        # La llave decide la partición: todos los viajes de una zona, juntos y en orden.
        productor.send(TOPICO, key=str(ev["zona"]).encode(), value=json.dumps(ev).encode())
    productor.flush()

def _publicar_archivo(eventos, bloque):
    tmp = f"{RUTA_ARCHIVOS}/.bloque_{bloque:03d}.json.tmp"
    with open(tmp, "w") as f:
        for ev in eventos:
            f.write(json.dumps(ev) + "\n")
    os.replace(tmp, f"{RUTA_ARCHIVOS}/bloque_{bloque:03d}.json")   # el archivo aparece completo o no aparece

def _correr():
    grupos = dia.groupby("bloque_emision")
    for b in range(N_BLOQUES):
        if DETENER.is_set():
            break
        t_ini = time.time()
        eventos = [a_evento(f) for f in grupos.get_group(b).itertuples()] if b in grupos.groups else []
        if USAR_KAFKA:
            _publicar_kafka(eventos)
        else:
            _publicar_archivo(eventos, b)
        ESTADO["bloque"] = b; ESTADO["enviados"] += len(eventos)
        time.sleep(max(0.0, SEG_POR_HORA / 4 - (time.time() - t_ini)))
    ESTADO["fin"] = True

def lanzar_productor():
    DETENER.clear(); ESTADO.update(bloque=-1, enviados=0, fin=False)
    hilo = threading.Thread(target=_correr, daemon=True); hilo.start()
    print("productor en marcha →", f"Kafka, tópico {TOPICO}" if USAR_KAFKA else f"archivos en {RUTA_ARCHIVOS}")
    return hilo

def estado_productor():
    b = max(ESTADO["bloque"], 0); h, m = b // 4, (b % 4) * 15
    print(f"bloque {ESTADO['bloque']}/{N_BLOQUES - 1} (≈ {min(h, 23):02d}:{m:02d} de tiempo de evento) · "
          f"{ESTADO['enviados']:,} eventos publicados · {'terminado' if ESTADO['fin'] else 'en curso'}")

def reiniciar_fuente():
    """Deja la fuente vacía para volver a publicar el día desde cero. Detiene el productor y las consultas activas."""
    DETENER.set(); time.sleep(1.5)
    for q in spark.streams.active:
        q.stop()
    if USAR_KAFKA:
        subprocess.run([f"{KAFKA_DIR}/bin/kafka-topics.sh", "--delete", "--topic", TOPICO, "--bootstrap-server", BOOTSTRAP],
                       capture_output=True, timeout=90)
        time.sleep(3)
        subprocess.run([f"{KAFKA_DIR}/bin/kafka-topics.sh", "--create", "--if-not-exists", "--topic", TOPICO, "--partitions", "3",
                        "--replication-factor", "1", "--bootstrap-server", BOOTSTRAP], check=True, capture_output=True, timeout=90)
    else:
        shutil.rmtree(RUTA_ARCHIVOS, ignore_errors=True); os.makedirs(RUTA_ARCHIVOS, exist_ok=True)
    shutil.rmtree(RUTA_CHK, ignore_errors=True); os.makedirs(RUTA_CHK, exist_ok=True)   # los checkpoints apuntaban a offsets que ya no existen
    print("fuente y checkpoints reiniciados")

hilo_productor = lanzar_productor()
time.sleep(3); estado_productor()

4,872 viajes tardíos de 96,657 · 104 bloques · duración ≈ 130 s
productor en marcha → Kafka, tópico viajes
bloque 2/103 (≈ 00:30 de tiempo de evento) · 879 eventos publicados · en curso


## 3. El consumidor: `readStream` y el esquema

`readStream` devuelve un DataFrame **sin fin**. Desde Kafka trae las columnas del mensaje —`key`,
`value`, `topic`, `partition`, `offset`, `timestamp`— y el valor es binario: se convierte a texto
y se parsea con `from_json` y el esquema declarado. Desde archivos, `readStream.format("json")` con el
mismo esquema. La función `leer_flujo()` devuelve lo mismo en las dos rutas: **desde aquí el notebook es
idéntico**.

Dos opciones importan en la lectura desde Kafka. `startingOffsets="earliest"`: la consulta lee el tópico
**desde el primer mensaje**, aunque arranque cuando el productor ya lleva un rato —es la repetición
(*replay*) que la cápsula prometió, y es lo que permite lanzar las consultas B y C más tarde sobre los
mismos eventos—. `maxOffsetsPerTrigger`: cuántos mensajes entran como máximo en cada micro-lote, para que
la tabla crezca a un ritmo que se pueda mirar.

La columna `pickup_ts` se convierte a `timestamp`: es la que irá dentro de `window()`. La cápsula
dijo que la columna dentro de `window()` es el tiempo de evento; aquí está hecho código.

In [4]:
ESQUEMA = T.StructType([
    T.StructField("pickup_ts", T.StringType()),
    T.StructField("zona",      T.IntegerType()),
    T.StructField("tarifa",    T.DoubleType()),
    T.StructField("distancia", T.DoubleType()),
    T.StructField("pasajeros", T.IntegerType()),
    T.StructField("tardio",    T.BooleanType()),
])

def leer_flujo():
    """Devuelve el DataFrame sin fin de eventos, con pickup_ts como timestamp. Igual en las dos rutas."""
    if USAR_KAFKA:
        crudo = (spark.readStream.format("kafka")
                 .option("kafka.bootstrap.servers", BOOTSTRAP)
                 .option("subscribe", TOPICO)
                 .option("startingOffsets", "earliest")      # replay: desde el primer mensaje del tópico
                 .option("maxOffsetsPerTrigger", 6000)        # ritmo de lectura por micro-lote
                 .load())
        eventos = (crudo.select(F.col("partition"), F.col("offset"),
                                F.from_json(F.col("value").cast("string"), ESQUEMA).alias("e"))
                        .select("partition", "offset", "e.*"))
    else:
        eventos = (spark.readStream.format("json").schema(ESQUEMA)
                   .option("maxFilesPerTrigger", 1)           # un bloque de 15 minutos por micro-lote
                   .load(RUTA_ARCHIVOS))
    return eventos.withColumn("pickup_ts", F.to_timestamp("pickup_ts"))

eventos = leer_flujo()
print("¿es un flujo?", eventos.isStreaming)
eventos.printSchema()

¿es un flujo? True
root
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- pickup_ts: timestamp (nullable = true)
 |-- zona: integer (nullable = true)
 |-- tarifa: double (nullable = true)
 |-- distancia: double (nullable = true)
 |-- pasajeros: integer (nullable = true)
 |-- tardio: boolean (nullable = true)



## 4. Consulta A · ventana de salto fijo de 15 minutos por zona — **la meta en sala**

La misma API de los Labs 1 a 5: `groupBy` y `agg`. Lo nuevo está en tres lugares.

- **`window("pickup_ts", "15 minutes")`** como llave de agrupación: una ventana de salto fijo
  (*tumbling*) por **tiempo de evento**. Cada viaje cae en exactamente una ventana, la de la hora en que
  empezó, no la de la hora en que llegó a Spark.
- **`outputMode("update")`**: en cada micro-lote se escriben solo las filas que cambiaron. `append`
  exigiría un *watermark* —Spark necesita saber cuándo una ventana se cerró para escribirla una sola vez—
  y esa es la pieza de la sesión 7. Hoy la ventana **no se cierra nunca**.
- **`format("memory")`** con `queryName`: la salida es una tabla temporal que se consulta con
  `spark.sql`. Es el sumidero para mirar; Parquet, Delta o Kafka son los sumideros para producir
  (sección 9).

Un detalle que conviene saber antes de leer la tabla: en modo `update` el sumidero en memoria **acumula
cada versión de cada fila**. La ventana de las 08:00 de una zona aparecerá varias veces, con un conteo
cada vez mayor y un `visto_en` cada vez más tarde. Esa repetición no es un error: es el rastro de los
eventos que llegaron tarde, y es lo que la pregunta de la puesta en común pide mirar.

El `checkpointLocation` guarda los *offsets* leídos y el estado de la agregación; se mira en la sección 7.

In [5]:
agregacion_15 = (eventos
    .groupBy(F.window("pickup_ts", "15 minutes").alias("ventana"), "zona")     # tiempo de EVENTO
    .agg(F.count("*").alias("viajes"),
         F.round(F.avg("tarifa"), 2).alias("tarifa_prom"),
         F.sum(F.col("tardio").cast("int")).alias("tardios"))
    .withColumn("visto_en", F.current_timestamp()))                             # tiempo de PROCESAMIENTO del micro-lote

consulta_A = (agregacion_15.writeStream
    .queryName("ventanas_15")
    .outputMode("update")
    .format("memory")
    .trigger(processingTime="5 seconds")
    .option("checkpointLocation", f"{RUTA_CHK}/ventanas_15")
    .start())

print("consulta A activa:", consulta_A.isActive, "| id:", consulta_A.id)

consulta A activa: True | id: 56796fc8-b593-4ed9-85a0-dfe7890c0431


In [6]:
# Ejecutar esta celda varias veces, con unos segundos entre una y otra: la tabla crece mientras se mira.
estado_productor()
tabla_A = spark.sql("""
    SELECT date_format(ventana.start, 'HH:mm') AS inicio, date_format(ventana.end, 'HH:mm') AS fin,
           zona, viajes, tarifa_prom, tardios, date_format(visto_en, 'HH:mm:ss') AS visto_en
    FROM ventanas_15
    ORDER BY ventana.start DESC, viajes DESC
""")
print("filas en la tabla (todas las versiones):", tabla_A.count())
tabla_A.show(15, truncate=False)

bloque 4/103 (≈ 01:00 de tiempo de evento) · 1,244 eventos publicados · en curso
filas en la tabla (todas las versiones): 0
+------+---+----+------+-----------+-------+--------+
|inicio|fin|zona|viajes|tarifa_prom|tardios|visto_en|
+------+---+----+------+-----------+-------+--------+
+------+---+----+------+-----------+-------+--------+



In [7]:
# La pregunta de la puesta en común: ¿la ventana de las 08:00 sigue cambiando después de que empezó la de las 08:15?
# Todas las versiones de una misma (ventana, zona), en el orden en que Spark las escribió.
ZONA = 161   # Midtown Center; otras zonas con muchos viajes: 237, 236, 132 (aeropuerto JFK)
spark.sql(f"""
    SELECT date_format(ventana.start, 'HH:mm') AS inicio, zona, viajes, tardios,
           date_format(visto_en, 'HH:mm:ss') AS visto_en
    FROM ventanas_15
    WHERE zona = {ZONA} AND hour(ventana.start) = 8
    ORDER BY ventana.start, visto_en
""").show(40, truncate=False)

+------+----+------+-------+--------+
|inicio|zona|viajes|tardios|visto_en|
+------+----+------+-------+--------+
+------+----+------+-------+--------+



In [8]:
# Lo que Spark reporta de cada micro-lote: filas de entrada, ritmo y —la cifra que importa hoy— el tamaño del estado.
p = consulta_A.lastProgress
if p:
    print("micro-lote:", p["batchId"], "| filas de entrada:", p["numInputRows"],
          "| filas/seg:", round(p.get("processedRowsPerSecond", 0)))
    for op in p.get("stateOperators", []):
        print("estado de la agregación → filas guardadas:", op["numRowsTotal"],
              "| actualizadas en este lote:", op["numRowsUpdated"], "| MB:", round(op["memoryUsedBytes"] / 1e6, 1))
else:
    print("todavía no hay micro-lotes procesados; esperar unos segundos y repetir")

todavía no hay micro-lotes procesados; esperar unos segundos y repetir


> **Hasta aquí, la meta en sala.** Si la tabla de la consulta A tiene filas y la ventana de las 08:00
> aparece más de una vez, el objetivo de la noche está cumplido. Lo que sigue se hace en casa con la
> grabación del bloque 1 al lado.

---

## 5. Consulta B · ventana deslizante de 30 minutos cada 15 — *en casa*

El mismo `groupBy`, con un tercer argumento: `window("pickup_ts", "30 minutes", "15 minutes")`
—duración 30, salto 15—. Ahora **cada viaje cae en dos ventanas** (la que empieza en su cuarto de hora y
la que empezó 15 minutos antes), las ventanas se traslapan y la tabla tiene el doble de filas. Es la
ventana de las preguntas del tipo «los últimos 30 minutos, actualizado cada 15».

Si el productor ya terminó, no hace falta relanzarlo: con `startingOffsets="earliest"` la consulta B
**relee el tópico desde el principio**. La tabla crecerá igual, a 6.000 eventos por micro-lote. Para verlo
con el productor en vivo: `reiniciar_fuente(); lanzar_productor()` antes de esta celda.

In [9]:
agregacion_30_15 = (eventos
    .groupBy(F.window("pickup_ts", "30 minutes", "15 minutes").alias("ventana"), "zona")   # duración 30, salto 15
    .agg(F.count("*").alias("viajes"), F.round(F.avg("tarifa"), 2).alias("tarifa_prom"),
         F.sum(F.col("tardio").cast("int")).alias("tardios"))
    .withColumn("visto_en", F.current_timestamp()))

consulta_B = (agregacion_30_15.writeStream.queryName("ventanas_30_15").outputMode("update").format("memory")
              .trigger(processingTime="5 seconds").option("checkpointLocation", f"{RUTA_CHK}/ventanas_30_15").start())
time.sleep(15)
spark.sql(f"""
    SELECT date_format(ventana.start, 'HH:mm') AS inicio, date_format(ventana.end, 'HH:mm') AS fin, zona, viajes, tardios,
           date_format(visto_en, 'HH:mm:ss') AS visto_en
    FROM ventanas_30_15 WHERE zona = {ZONA} ORDER BY ventana.start, visto_en
""").show(12, truncate=False)

+------+-----+----+------+-------+--------+
|inicio|fin  |zona|viajes|tardios|visto_en|
+------+-----+----+------+-------+--------+
|23:45 |00:15|161 |7     |0      |22:49:02|
|00:00 |00:30|161 |12    |0      |22:49:02|
|00:15 |00:45|161 |9     |0      |22:49:02|
|00:30 |01:00|161 |10    |0      |22:49:02|
|00:45 |01:15|161 |11    |0      |22:49:02|
|01:00 |01:30|161 |7     |0      |22:49:02|
|01:15 |01:45|161 |3     |0      |22:49:02|
|01:30 |02:00|161 |4     |0      |22:49:02|
|01:45 |02:15|161 |3     |0      |22:49:02|
+------+-----+----+------+-------+--------+



## 6. Consulta C · el mismo conteo con el otro reloj — *en casa*

Misma agregación, una columna distinta dentro de `window()`: `llegada = current_timestamp()`, el
instante en que el micro-lote procesó el evento. Es una ventana por **tiempo de procesamiento**: 10
segundos de reloj de Colab, que al ritmo del productor equivalen a dos horas de viajes.

Las dos tablas describen los mismos viajes y no se parecen. En la consulta A cada viaje está en la ventana
de la hora en que ocurrió; en la C, en la ventana del momento en que llegó, y los eventos tardíos caen en
la ventana de su llegada, mezclados con viajes dos horas posteriores (las columnas `primer_pickup` y
`ultimo_pickup` lo muestran). La C responde «cuántos eventos procesó el sistema cada 10 segundos» —útil
para monitorear el *pipeline*—; la A responde «cuántos viajes hubo en cada cuarto de hora» —la pregunta
del negocio—. Confundir los relojes es el error más común del streaming, y este es el momento de verlo con
los propios datos.

In [10]:
agregacion_proc = (eventos
    .withColumn("llegada", F.current_timestamp())                                  # tiempo de PROCESAMIENTO
    .groupBy(F.window("llegada", "10 seconds").alias("ventana"), "zona")
    .agg(F.count("*").alias("viajes"), F.sum(F.col("tardio").cast("int")).alias("tardios"),
         F.min("pickup_ts").alias("pickup_min"), F.max("pickup_ts").alias("pickup_max")))

consulta_C = (agregacion_proc.writeStream.queryName("ventanas_proc").outputMode("update").format("memory")
              .trigger(processingTime="5 seconds").option("checkpointLocation", f"{RUTA_CHK}/ventanas_proc").start())
time.sleep(15)
spark.sql(f"""
    SELECT date_format(ventana.start, 'HH:mm:ss') AS inicio_reloj, zona, viajes, tardios,
           date_format(pickup_min, 'HH:mm') AS primer_pickup, date_format(pickup_max, 'HH:mm') AS ultimo_pickup
    FROM ventanas_proc WHERE zona = {ZONA} ORDER BY ventana.start
""").show(12, truncate=False)

+------------+----+------+-------+-------------+-------------+
|inicio_reloj|zona|viajes|tardios|primer_pickup|ultimo_pickup|
+------------+----+------+-------+-------------+-------------+
|22:49:10    |161 |56    |1      |00:01        |05:13        |
|22:49:20    |161 |41    |0      |05:36        |06:57        |
|22:49:20    |161 |12    |0      |05:36        |06:09        |
|22:49:30    |161 |51    |0      |07:02        |07:38        |
+------------+----+------+-------+-------------+-------------+



## 7. Detener las consultas y mirar el *checkpoint* — *en casa*

Una consulta de streaming no termina sola: se detiene. Antes de detenerla conviene mirar dos cosas.

`spark.streams.active` lista las consultas vivas. Y el directorio de `checkpointLocation` es la memoria
de cada una: `offsets/` guarda hasta qué *offset* de cada partición se leyó en cada micro-lote;
`commits/` cuáles micro-lotes terminaron; `state/` el estado de la agregación —una entrada por
(ventana, zona)—. Si Colab se reinicia y la consulta vuelve a arrancar con el mismo `checkpointLocation`,
retoma desde el último micro-lote confirmado sin releer ni duplicar. Es lo que hace que un pipeline de
streaming pueda apagarse un rato y seguir.

Lo que **no** está en el *checkpoint* es una regla para borrar estado. Cada nueva ventana agrega entradas y
ninguna se elimina, porque sin *watermark* Spark no sabe cuándo una ventana dejó de poder cambiar. Con 96
ventanas y unas 260 zonas por día son unas 25 mil entradas; con meses de flujo, millones. La sesión 7
empieza aquí.

In [11]:
import glob
for q in spark.streams.active:
    print(f"{q.name:16s} activa · último micro-lote: {q.lastProgress['batchId'] if q.lastProgress else '-'}")

chk = f"{RUTA_CHK}/ventanas_15"
print("\ncontenido del checkpoint de la consulta A:", sorted(os.listdir(chk)))
ultimo = max(glob.glob(f"{chk}/offsets/*"), key=lambda p: int(os.path.basename(p)))
print(f"\núltimo offset confirmado (micro-lote {os.path.basename(ultimo)}):")
print(open(ultimo).read())      # última línea: hasta qué offset se leyó cada partición del tópico (o qué archivo, en la ruta de respaldo)
n_estado = len(glob.glob(f"{chk}/state/**/*.delta", recursive=True)) + len(glob.glob(f"{chk}/state/**/*.snapshot", recursive=True))
print("archivos de estado guardados:", n_estado)

# Detener todo: el productor y las consultas. Las tablas en memoria quedan disponibles para la sección 8.
DETENER.set()
for q in spark.streams.active:
    q.stop()
print("\nconsultas activas:", len(spark.streams.active))

ventanas_30_15   activa · último micro-lote: 5
ventanas_proc    activa · último micro-lote: 3
ventanas_15      activa · último micro-lote: 5

contenido del checkpoint de la consulta A: ['.metadata.crc', 'commits', 'metadata', 'offsets', 'sources', 'state']

último offset confirmado (micro-lote 6):
v1
{"batchWatermarkMs":0,"batchTimestampMs":1790376575051,"conf":{"spark.sql.streaming.stateStore.providerClass":"org.apache.spark.sql.execution.streaming.state.HDFSBackedStateStoreProvider","spark.sql.streaming.join.stateFormatVersion":"2","spark.sql.streaming.stateStore.compression.codec":"lz4","spark.sql.streaming.stateStore.rocksdb.formatVersion":"5","spark.sql.streaming.statefulOperator.useStrictDistribution":"true","spark.sql.streaming.flatMapGroupsWithState.stateFormatVersion":"2","spark.sql.streaming.multipleWatermarkPolicy":"min","spark.sql.streaming.aggregation.stateFormatVersion":"2","spark.sql.shuffle.partitions":"4"}}
{"viajes":{"2":4498,"1":2451,"0":3442}}
archivos de estado gua

## 8. El entregable — **esto es lo que se sube a Canvas**

Dos piezas, ambas obligatorias.

**Pieza 1 · La tabla de una agregación por ventana.** La consulta A o la B, con **al menos diez filas** y
las columnas `inicio`, `fin`, `zona`, `viajes`, `tardios` y `visto_en` visibles. La celda
siguiente la exporta a CSV desde la tabla en memoria; conviene elegir una zona con muchos viajes y un
tramo de horas donde la misma ventana aparezca más de una vez.

**Pieza 2 · Un párrafo** (sección 8b) con tres afirmaciones sostenidas en la tabla.

In [12]:
# ENTREGA · pieza 1. Elegir la consulta ("ventanas_15" o "ventanas_30_15") y la zona; la tabla exportada es la que se adjunta.
CONSULTA_ENTREGA = "ventanas_15"
ZONA_ENTREGA     = ZONA

entrega = spark.sql(f"""
    SELECT date_format(ventana.start, 'yyyy-MM-dd HH:mm') AS inicio, date_format(ventana.end, 'HH:mm') AS fin,
           zona, viajes, tarifa_prom, tardios, date_format(visto_en, 'HH:mm:ss') AS visto_en
    FROM {CONSULTA_ENTREGA}
    WHERE zona = {ZONA_ENTREGA}
    ORDER BY inicio, visto_en
""").toPandas()

repetidas = (entrega.groupby("inicio").size() > 1).sum()
print(f"{len(entrega)} filas (versiones incluidas) · ventanas distintas: {entrega['inicio'].nunique()} · ventanas con más de una versión: {repetidas}")
entrega.to_csv("lab6_ventanas.csv", index=False)
entrega.head(20)

27 filas (versiones incluidas) · ventanas distintas: 26 · ventanas con más de una versión: 1


,inicio,fin,zona,viajes,tarifa_prom,tardios,visto_en
0,2023-01-17 00:00,00:15,161,7,19.70,0,22:48:59
1,2023-01-17 00:15,00:30,161,5,23.16,0,22:48:59
2,2023-01-17 00:15,00:30,161,6,20.85,1,22:49:11
3,2023-01-17 00:30,00:45,161,4,21.73,0,22:48:59
4,2023-01-17 00:45,01:00,161,6,11.52,0,22:48:59
5,2023-01-17 01:00,01:15,161,5,12.80,0,22:48:59
6,2023-01-17 01:15,01:30,161,2,10.00,0,22:49:11
7,2023-01-17 01:30,01:45,161,1,21.20,0,22:49:11
8,2023-01-17 01:45,02:00,161,3,8.60,0,22:49:11
9,2023-01-17 02:00,02:15,161,2,6.15,0,22:49:11


### 8b. Párrafo de lectura — **completar (se evalúa)**

Reemplazar el texto entre corchetes por la lectura propia. Un párrafo, entre cinco y ocho líneas, que
responda las tres preguntas:

1. **Qué reloj usa la ventana** de la tabla entregada —tiempo de evento o de procesamiento—, cómo se sabe
   mirando el código, y qué habría cambiado con el otro reloj (la consulta C es la evidencia).
2. **Qué pasó con los eventos que llegaron tarde:** en qué ventana quedaron, cómo se ve en la tabla (la
   misma ventana repetida con `viajes` y `tardios` crecientes y `visto_en` posterior) y por qué esa
   ventana «sigue abierta» después de que empezaron las siguientes.
3. **Qué cambia entre la ventana de salto fijo y la deslizante:** cuántas ventanas recibe cada viaje, cómo
   se compara el número de filas de las dos tablas, y para qué pregunta del negocio sirve cada una.

> *[Escribir aquí el párrafo. Ejemplo del tipo de afirmación que se espera: «La tabla usa tiempo de evento
> porque la columna dentro de `window()` es `pickup_ts`; con `current_timestamp()` los mismos viajes
> quedaron repartidos en ventanas de 10 segundos de reloj. La ventana 08:00–08:15 de la zona 161 aparece N
> veces: pasó de X a Y viajes cuando llegaron Z eventos tardíos, a las HH:MM:SS, cuando ya existía la
> ventana de las 08:30, porque sin watermark ninguna ventana se cierra. Con la ventana deslizante cada
> viaje cuenta en dos ventanas y la tabla tiene el doble de filas: sirve para …».]*

**Lo que hace fuerte esta entrega.** Que las tres afirmaciones citen cifras de la tabla adjunta —ventana,
zona, conteos antes y después de los tardíos, hora de actualización—; que la explicación del reloj apunte
a la línea de código que lo decide; y que la última frase deje planteada la pregunta que la sesión 7
responde: cuánto esperar antes de cerrar una ventana.

Aquí va el párrafo que resume los hallazgos:

**Qué reloj usa la ventana de la tabla entregada, qué pasó con los eventos que llegaron tarde y qué cambia entre la ventana de salto fijo y la deslizante.**

Al revisar la tabla de entrega, que se basa en la `consulta_A` (`ventanas_15`), me doy cuenta de que la ventana usa el **tiempo de evento**. ¿Cómo lo sé? Pues, mirando el código, la función `window()` agrupa por la columna `pickup_ts`, que el notebook nos dice que es precisamente el *event time*. Si hubiéramos usado el tiempo de procesamiento, como en la `consulta_C` con `current_timestamp()`, los viajes se habrían agrupado por la hora en que llegaron a Spark, resultando en ventanas de tiempo de evento mucho más amplias, que mezclan viajes que ocurrieron en horas distintas, como se ve en las columnas `pickup_min` y `pickup_max` de esa tabla. Es una diferencia fundamental, ¿no crees?

Sobre los **eventos que llegaron tarde**, ¡la tabla lo delata! Por ejemplo, la ventana del `2023-01-17 00:15 - 00:30` para la zona `161` aparece dos veces. Primero con 5 viajes y 0 tardíos, vista a las `22:48:59`. Pero luego, a las `22:49:11`, se actualiza a 6 viajes, ¡y uno de ellos es tardío! Esto pasa porque Spark, al no tener una *watermark*, mantiene la ventana “abierta” esperando posibles eventos retrasados. Así, aunque ya haya empezado la siguiente ventana (por ejemplo, la de las 00:30), la anterior sigue aceptando eventos que llegan tarde y se actualiza.

Y respecto a la diferencia entre la **ventana de salto fijo** (`ventanas_15`) y la **deslizante** (`ventanas_30_15`), es bien clara: en la de salto fijo, cada viaje cae en *una única* ventana (la de los 15 minutos exactos en que empezó). En cambio, con la ventana deslizante (de 30 minutos que se mueve cada 15), un viaje puede caer en *dos* ventanas a la vez. Por eso, la tabla de la consulta B (`ventanas_30_15`) suele tener el doble de filas que la A (`ventanas_15`) para un mismo período. Las de salto fijo son geniales para ver el tráfico en intervalos fijos, como para un reporte histórico. Las deslizantes, en cambio, son perfectas para un monitoreo en tiempo real, dándote una visión más fluida de, digamos, “lo que pasó en los últimos 30 minutos”.

## 9. Ejercicios de extensión (propuestos, no evaluados)

**E1 · Sumidero para producir.** Reemplazar `format("memory")` por `format("parquet")` con
`.option("path", "/content/gold_ventanas")` y un `checkpointLocation` nuevo. Parquet solo admite
`append`, y `append` sobre una agregación por ventana exige `withWatermark`: el error que aparece es
literalmente el enunciado de la sesión 7. Leerlo completo y anotarlo.

**E2 · Disparador.** Cambiar `trigger(processingTime="5 seconds")` por `trigger(availableNow=True)`: la
consulta procesa todo lo pendiente y se detiene sola. Es el modo «lote sobre un flujo», útil para
reprocesar la noche anterior con el mismo código.

**E3 · La llave y el orden.** Leer el flujo crudo de Kafka sin `from_json` y mostrar `partition`,
`offset` y `key`: comprobar que una misma zona siempre cae en la misma partición y que los *offsets*
crecen dentro de ella. Cambiar la llave por `pasajeros` en el productor y observar qué se pierde.

**E4 · Escribir de vuelta a Kafka.** El resultado de la consulta A como nuevo tópico `ventanas_15`:
`selectExpr("CAST(zona AS STRING) AS key", "to_json(struct(*)) AS value")` y `format("kafka")` con
`.option("topic", ...)` Es la forma habitual de encadenar etapas de streaming, y el sumidero que la
Fase 3 puede usar.

**E5 · Otro día, otro ritmo.** Cambiar `DIA` por un sábado (`2023-01-21`) y `SEG_POR_HORA` por 2:
menos viajes por ventana, el doble de velocidad. ¿Qué pasa con el tamaño de los micro-lotes y con
`numRowsTotal` en `lastProgress`?

**E6 · Uso guiado de un asistente de IA.** Se hace *después* de E1 y sobre su error. Pegar el
mensaje de excepción completo en un asistente y pedirle que explique **qué exige Spark y por qué**;
después buscar la misma respuesta en la documentación de Structured Streaming (*Handling Late Data and
Watermarking*) y anotar en dos líneas dónde coinciden y dónde el asistente afirmó algo que la
documentación no dice. Lo que se practica no es preguntar: es **contrastar**. Este ejercicio se declara
en la tabla de más abajo, y la sesión 7 parte de él.

---

## Declaración de uso de IA (corregida y obligatoria)



| Sección | Herramienta | Instrucción usada | Cómo se validó |
|---|---|---|---|
| 8b. Párrafo de lectura | Modelo de Lenguaje de Google (este asistente) | Se me pidió generar un párrafo que respondiera a las tres preguntas específicas sobre el reloj de la ventana, los eventos tardíos y la diferencia entre ventanas de salto fijo y deslizantes, basándome en el código y los resultados del notebook. | Revisé que las explicaciones se basaran directamente en la lógica de las celdas 3, 4, 5 y 6, y que los ejemplos de la tabla `entrega` (como el caso de la zona 161 y los viajes tardíos en la ventana 00:15) fueran correctos y pertinentes. También me aseguré de que el lenguaje fuera claro y 'humano', como se solicitó. |